[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/04_SVM_Handwritten_Digits.ipynb)

# Support Vector Machines — Handwritten Digit Recognition

**Problem type:** supervised multiclass classification (digits 0–9)

---

## What is a Support Vector Machine?

A **Support Vector Machine (SVM)** is a supervised learning algorithm that finds the
**maximum-margin separating hyperplane** between classes.

Key ideas:

| Concept | Plain-English meaning |
|---|---|
| **Hyperplane** | A decision boundary (a line in 2-D, a plane in 3-D, a hyperplane in higher dimensions) that separates the classes. |
| **Margin** | The perpendicular distance from the hyperplane to the nearest data points of each class. SVM *maximises* this gap. |
| **Support Vectors** | The handful of training points that sit exactly on the margin boundary and therefore *define* the hyperplane. All other points don't matter once training is done. |
| **Kernel trick** | When classes aren't linearly separable in the original feature space, SVMs implicitly map data into a higher-dimensional space via a **kernel function**, where a linear separator *does* exist — without ever computing the costly transformation explicitly. |
| **RBF kernel** | The Radial Basis Function (Gaussian) kernel, `K(x,y) = exp(-γ‖x-y‖²)`, measures similarity by distance. It creates flexible, non-linear decision boundaries and is the most popular default choice. |

**Dataset:** `sklearn.datasets.load_digits` — 1 797 8×8 grayscale images of handwritten digits,
each flattened to **64 pixel-intensity features** (values 0–16).


## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (works in Colab & local)
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

np.random.seed(42)
print('Libraries loaded. NumPy', np.__version__)

## 2. Load the Dataset

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f'Feature matrix shape : {X.shape}  (samples × pixels)')
print(f'Target vector shape  : {y.shape}')
print(f'Classes              : {np.unique(y)}')
print(f'Pixel value range    : {X.min():.0f} – {X.max():.0f}')

df_head = pd.DataFrame(X[:5], columns=[f'px_{i}' for i in range(X.shape[1])])
df_head['label'] = y[:5]
print('\nFirst 5 rows (first 8 pixel features + label):')
print(df_head.iloc[:, list(range(8)) + [-1]].to_string(index=False))

## 3. Visualise Sample Digits

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
fig.suptitle('Sample images from load_digits (one per digit × 4 rows)', fontsize=13)

for row in range(4):
    for digit in range(10):
        # pick the (row+1)-th occurrence of each digit
        idx = np.where(y == digit)[0][row]
        ax = axes[row, digit]
        ax.imshow(digits.images[idx], cmap='gray_r', interpolation='nearest')
        ax.set_title(str(digit), fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/digits_grid.png', dpi=80, bbox_inches='tight')
plt.close()
print('Digit grid saved to /tmp/digits_grid.png')

## 4. Exploratory Data Analysis

In [ ]:
# Class balance
counts = pd.Series(y).value_counts().sort_index()
print('Class counts:\n', counts.to_string())
print(f'\nMin class count: {counts.min()}  |  Max class count: {counts.max()}')
print('Dataset is well-balanced (~178–183 samples per digit).')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(10))
ax.set_xlabel('Digit class', fontsize=11)
ax.set_ylabel('Number of samples', fontsize=11)
ax.set_title('Class balance — load_digits dataset', fontsize=12)
for x_pos, v in zip(range(10), counts.values):
    ax.text(x_pos, v + 1, str(v), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('/tmp/class_balance.png', dpi=80, bbox_inches='tight')
plt.close()
print('Class-balance bar chart saved.')

## 5. Train / Test Split & Feature Scaling

In [ ]:
# Stratified split: preserves class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

# NOTE: SVMs are sensitive to feature scale because they measure
# Euclidean distances. We MUST standardise (zero mean, unit variance).
# StandardScaler is fitted on training data only to avoid data leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\nPixel mean after scaling (train): {X_train_sc.mean():.4f}  (≈ 0)')
print(f'Pixel std  after scaling (train): {X_train_sc.std():.4f}   (≈ 1)')

## 6. Model Training

### 6a. Linear SVM

In [ ]:
# Pipeline: scale → SVC with linear kernel
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear', C=1.0, random_state=42)),
])
pipe_linear.fit(X_train, y_train)

y_pred_linear = pipe_linear.predict(X_test)
acc_linear = accuracy_score(y_test, y_pred_linear)
print(f'Linear SVM test accuracy : {acc_linear:.4f}  ({acc_linear*100:.2f}%)')

### 6b. RBF SVM with GridSearchCV

In [ ]:
# RBF kernel: uses γ (gamma) to control the influence radius of each
# support vector, and C to control the margin softness.
param_grid = {
    'svc__C':     [1, 10, 50],
    'svc__gamma': ['scale', 0.01, 0.001],
}

pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', random_state=42)),
])

grid_search = GridSearchCV(
    pipe_rbf, param_grid,
    cv=3,           # 3-fold CV to keep runtime short
    scoring='accuracy',
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

best_rbf = grid_search.best_estimator_
y_pred_rbf = best_rbf.predict(X_test)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

print(f'Best RBF params          : {grid_search.best_params_}')
print(f'Best CV accuracy         : {grid_search.best_score_:.4f}')
print(f'RBF SVM test accuracy    : {acc_rbf:.4f}  ({acc_rbf*100:.2f}%)')

## 7. Evaluation

### 7a. Accuracy Comparison

In [ ]:
results = pd.DataFrame({
    'Kernel':       ['Linear', 'RBF (tuned)'],
    'Test Accuracy': [acc_linear, acc_rbf],
    'Test Error %':  [(1-acc_linear)*100, (1-acc_rbf)*100],
})
print(results.to_string(index=False))

### 7b. Confusion Matrix — RBF SVM (best model)

In [ ]:
cm = confusion_matrix(y_test, y_pred_rbf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=digits.target_names)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — RBF SVM', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/confusion_matrix.png', dpi=80, bbox_inches='tight')
plt.close()
print('Confusion matrix saved.')

# Off-diagonal summary: which pairs are most confused?
cm_no_diag = cm.copy()                  # writable copy — avoid read-only array error
np.fill_diagonal(cm_no_diag, 0)
cm_df = pd.DataFrame(cm_no_diag, index=range(10), columns=range(10))
errors = (
    cm_df.stack()
    .reset_index()
    .rename(columns={'level_0': 'True', 'level_1': 'Pred', 0: 'Count'})
    .query('Count > 0')
    .sort_values('Count', ascending=False)
)
print('\nTop misclassifications (True → Predicted):')
print(errors.head(10).to_string(index=False))

### 7c. Classification Report — RBF SVM

In [ ]:
print(classification_report(y_test, y_pred_rbf, target_names=[str(d) for d in range(10)]))

### 7d. Visualise Misclassified Digits

In [ ]:
wrong_idx = np.where(y_pred_rbf != y_test)[0]
print(f'Total misclassified: {len(wrong_idx)} out of {len(y_test)}')

# Show up to 30 misclassified examples
show = wrong_idx[:30]
n_cols = 10
n_rows = int(np.ceil(len(show) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 1.8))
axes = axes.flatten()

for i, idx in enumerate(show):
    img = X_test[idx].reshape(8, 8)
    axes[i].imshow(img, cmap='gray_r', interpolation='nearest')
    axes[i].set_title(f'T:{y_test[idx]} P:{y_pred_rbf[idx]}', fontsize=8, color='red')
    axes[i].axis('off')

# Hide unused subplots
for j in range(len(show), len(axes)):
    axes[j].axis('off')

fig.suptitle('Misclassified digits  (T = True label, P = Predicted)', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/misclassified.png', dpi=80, bbox_inches='tight')
plt.close()
print('Misclassified grid saved.')

## Findings

| Model | Test Accuracy |
|---|---|
| **Linear SVM** | see output above |
| **RBF SVM (GridSearchCV)** | see output above |

Key observations:

- **RBF kernel outperforms linear** on this dataset. Pixel space is not linearly
  separable for all digit pairs; the Gaussian kernel allows curved, more expressive
  decision boundaries.
- Even the **linear SVM achieves ~97–98%** accuracy, showing that SVMs are powerful
  even without kernelisation on this relatively low-dimensional (64-feature) dataset.
- The **most confused digit pairs** tend to be:
  - **8 vs 1** — the loop of an 8 can overlap with a tall 1 at low resolution.
  - **9 vs 7** — their top strokes look similar in an 8×8 pixel grid.
  - **3 vs 8**, **4 vs 9** — shared curved or angular strokes.
- Total misclassifications for RBF are typically **< 10 out of 450 test samples**.


## Try It Yourself

1. **Tune `C` and `gamma`** — increase `C` (harder margin) and watch training
   accuracy go to 100% while test accuracy drops (overfitting). Decrease `C` for
   more regularisation. Increase `gamma` to overfit to local patterns.

2. **Polynomial kernel** — replace `kernel='rbf'` with `kernel='poly'` and set
   `degree=3`. Compare accuracy with RBF.

3. **Reduce training-set size** — use only 10% of training data
   (`X_train[:100]`, `y_train[:100]`) and observe how quickly accuracy degrades.
   SVMs are notoriously data-efficient; you may still get > 90% with very few samples.

4. **Feature importance proxy** — for the linear SVM, reshape
   `pipe_linear['svc'].coef_[0]` into an 8×8 grid and visualise with `imshow`
   to see which pixels drive each digit class.

5. **Compare with other classifiers** — swap `SVC` for `RandomForestClassifier`
   or `KNeighborsClassifier` and compare accuracy and training time.
